[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C66_Agentic_Evaluation_Course/03_trajectory/03_trajectory_eval.ipynb)

# 03 · 轨迹级与过程级评测（schema / 六指标 / 循环检测 / LCS / 失败分类 / kappa / Goodhart）

目标：把「它是怎么做到的」从一堆日志，变成**六个可以算、可以比、可以进报告的数字**。

本 notebook 你会亲手实现：
1. **轨迹 schema 与模拟器** —— 生成带工具错误、冗余调用、循环的真实感轨迹
2. **六个轨迹指标** —— 步数分位数、工具精确/召回、无效动作率、冗余率、恢复率
3. **循环检测** —— 精确循环（滑窗哈希）与语义循环（状态摘要不变）
4. **轨迹比较** —— 编辑距离与 LCS；必经动作的子序列断言
5. **失败模式自动分类器** —— 互斥有优先级的七分类
6. **过程标注一致性** —— Cohen's kappa，以及「换个问法就能提高一致性」
7. **Goodhart 演示** —— 优化平均步数如何把成功率优化没了

> 心智模型：**结果层告诉你成没成，轨迹层告诉你钱花在哪、卡在哪、能不能自救。
> 而所有效率指标都必须分层到「成功轨迹」上，否则最省事的优化路径永远是「早点放弃」。**

## 1 · 轨迹 schema 与模拟器

先把讲解里的 schema 变成代码，并生成一批带真实病灶（工具错误、冗余调用、循环）的轨迹。

In [ ]:
import math, json, hashlib
from collections import Counter, defaultdict
import numpy as np

TOOLS = ['search_orders', 'get_policy', 'update_order', 'send_email', 'list_flights']

def make_step(i, tool, args, status='ok', error_code=None, tokens=(1200, 60)):
    return {'i': i, 'type': 'tool_call', 'tool': tool,
            'args_hash': hashlib.md5(json.dumps(args, sort_keys=True).encode()).hexdigest()[:8],
            'status': status, 'error_code': error_code,
            'tokens_in': tokens[0], 'tokens_out': tokens[1],
            'state_digest': None}

def simulate_trajectory(rng, style='healthy', required_tools=('search_orders', 'update_order'),
                        max_steps=30, err_rate=0.15, recover_p=0.85):
    """生成一条轨迹。style: healthy | looper | quitter | tool_misuser"""
    steps, state = [], 0
    i = 0
    done = False
    budget = {'healthy': max_steps, 'looper': max_steps,
              'quitter': int(rng.integers(2, 5)), 'tool_misuser': max_steps}[style]
    while i < budget:
        if style == 'looper' and i >= 3:
            tool, args = 'search_orders', {'q': 'refund'}           # 反复同样的查询
        elif style == 'tool_misuser':
            tool, args = 'update_order', {'id': int(rng.integers(0, 1000))}
        else:
            tool = required_tools[min(i, len(required_tools) - 1)] if i < len(required_tools) \
                else str(rng.choice(TOOLS))
            args = {'q': int(rng.integers(0, 1000))}
        err = rng.random() < (0.6 if style == 'tool_misuser' else err_rate)
        code = 'E_BAD_ARGS' if style == 'tool_misuser' else             str(rng.choice(['E_TIMEOUT', 'E_RATE_LIMIT', 'E_BAD_ARGS']))
        st = make_step(i, tool, args,
                       status='tool_error' if err else 'ok',
                       error_code=code if err else None)
        if not err and style != 'looper':
            state += 1
        st['state_digest'] = f'S{state}'
        steps.append(st)
        i += 1
        if err and rng.random() > recover_p and style == 'healthy':
            break                                                   # 没恢复，直接崩掉
        if style == 'healthy' and state >= 4 and rng.random() < 0.5:
            done = True
            break
    return {'run_id': 'demo', 'task_id': 't0', 'attempt': 0,
            'scaffold_version': 'harness@1.7.2', 'steps': steps,
            'outcome': {'score': int(done), 'reason': None if done else 'incomplete'},
            'max_steps': max_steps, 'style': style}

rng = np.random.default_rng(0)
demo = simulate_trajectory(rng, 'healthy')
print('一条轨迹的前 3 步:')
for s in demo['steps'][:3]:
    print(' ', {k: s[k] for k in ('i', 'tool', 'args_hash', 'status', 'state_digest')})
print(f"\n总步数 {len(demo['steps'])} | 结果 {demo['outcome']}")
assert all('args_hash' in s and 'status' in s for s in demo['steps'])
print('\n✅ schema 就位。注意 args_hash 与 status 两个字段——')
print('   没有它们，冗余率与恢复率这两个最有用的指标就永远算不出来。')

## 2 · 六个轨迹指标

In [ ]:
def steps_quantiles(trajs, successful_only=True):
    """效率指标只在成功轨迹上算——这是第 7 节 Goodhart 规则的第一条。"""
    sel = [t for t in trajs if (t['outcome']['score'] == 1 or not successful_only)]
    if not sel:
        return {'n': 0}
    lens = np.array([len(t['steps']) for t in sel])
    return {'n': len(sel), 'median': float(np.median(lens)),
            'p90': float(np.percentile(lens, 90)), 'mean': float(lens.mean())}

def tool_precision_recall(traj, required_tools):
    used = [s['tool'] for s in traj['steps']]
    used_set, req_set = set(used), set(required_tools)
    prec = len([u for u in used if u in req_set]) / len(used) if used else 0.0
    rec = len(used_set & req_set) / len(req_set) if req_set else 1.0
    return prec, rec

def invalid_action_rate(traj):
    steps = traj['steps']
    return sum(1 for s in steps if s['status'] != 'ok') / len(steps) if steps else 0.0

def redundancy_rate(traj):
    keys = [(s['tool'], s['args_hash']) for s in traj['steps']]
    seen, dup = set(), 0
    for k in keys:
        if k in seen:
            dup += 1
        seen.add(k)
    return dup / len(keys) if keys else 0.0

def recovery_rate(trajs):
    """出现过工具错误的轨迹中，最终成功的比例（条件成功率）。"""
    with_err = [t for t in trajs if any(s['status'] != 'ok' for s in t['steps'])]
    if not with_err:
        return float('nan')
    return float(np.mean([t['outcome']['score'] for t in with_err]))

rng = np.random.default_rng(4)
healthy = [simulate_trajectory(rng, 'healthy') for _ in range(300)]
loopers = [simulate_trajectory(rng, 'looper') for _ in range(80)]
quitters = [simulate_trajectory(rng, 'quitter') for _ in range(80)]

print('成功轨迹的步数分布:', steps_quantiles(healthy))
print('全部轨迹的步数分布:', steps_quantiles(healthy, successful_only=False))
p, r = tool_precision_recall(healthy[0], ('search_orders', 'update_order'))
print(f'\n单条轨迹 工具精确率 {p:.2f} 召回率 {r:.2f}')
print(f'无效动作率(healthy 均值) {np.mean([invalid_action_rate(t) for t in healthy]):.1%}')
print(f'冗余率 healthy {np.mean([redundancy_rate(t) for t in healthy]):.1%} '
      f'| looper {np.mean([redundancy_rate(t) for t in loopers]):.1%}')
assert np.mean([redundancy_rate(t) for t in loopers]) > 3 * np.mean([redundancy_rate(t) for t in healthy])
print('\n✅ 冗余率把 looper 和 healthy 拉开了三倍以上——')
print('   而这两组的「成功率」可能完全一样（都失败），结果层完全看不出区别。')

In [ ]:
# 条件成功率：有错误 vs 无错误
def conditional_success(trajs):
    with_err = [t['outcome']['score'] for t in trajs if any(s['status'] != 'ok' for s in t['steps'])]
    no_err = [t['outcome']['score'] for t in trajs if all(s['status'] == 'ok' for s in t['steps'])]
    return (float(np.mean(with_err)) if with_err else float('nan'),
            float(np.mean(no_err)) if no_err else float('nan'))

we, ne = conditional_success(healthy)
overall = float(np.mean([t['outcome']['score'] for t in healthy]))
print(f'总成功率        {overall:.1%}')
print(f'  出过错的轨迹  {we:.1%}   ← 这个数字才预测得了线上表现')
print(f'  没出错的轨迹  {ne:.1%}')
assert ne > we, '出过错的轨迹成功率必然更低'
gap = ne - we
print(f'\n落差 {gap:.1%}。线上环境的错误率通常高于离线环境，')
print('所以线上成功率会向「出过错的轨迹」那一档滑落——这就是 demo 与产品之间的鸿沟。')

## 3 · 循环检测：精确循环与语义循环

In [ ]:
def detect_exact_loop(traj, window=3, repeats=2):
    """精确循环：长度为 window 的动作序列连续重复 repeats 次以上。"""
    keys = [(s['tool'], s['args_hash']) for s in traj['steps']]
    n = len(keys)
    for start in range(n - window * repeats + 1):
        block = keys[start:start + window]
        if all(keys[start + w * window:start + (w + 1) * window] == block for w in range(repeats)):
            return True, start
    return False, None

def detect_semantic_loop(traj, k=4):
    """语义循环：状态摘要连续 k 步没有变化。"""
    digs = [s['state_digest'] for s in traj['steps']]
    run = 1
    for a, b in zip(digs, digs[1:]):
        run = run + 1 if a == b else 1
        if run >= k:
            return True
    return False

n_loop_exact = sum(detect_exact_loop(t)[0] for t in loopers)
n_health_exact = sum(detect_exact_loop(t)[0] for t in healthy)
n_loop_sem = sum(detect_semantic_loop(t) for t in loopers)
n_health_sem = sum(detect_semantic_loop(t) for t in healthy)
print(f'精确循环检出  looper {n_loop_exact}/{len(loopers)} | healthy {n_health_exact}/{len(healthy)}')
print(f'语义循环检出  looper {n_loop_sem}/{len(loopers)} | healthy {n_health_sem}/{len(healthy)}')
assert n_loop_exact / len(loopers) > 0.8
assert n_health_exact / len(healthy) < 0.2
print('\n✅ 检出率 >80%、误报率 <20%。把它接到预算控制上：检出即终止，')
print('   省下的预算拿去跑别的任务。注意被终止的轨迹要单独标记，不能简单记成失败。')

## 4 · 轨迹比较：编辑距离、LCS、必经动作断言

In [ ]:
def edit_distance(a, b):
    n, m = len(a), len(b)
    dp = list(range(m + 1))
    for i in range(1, n + 1):
        prev, dp[0] = dp[0], i
        for j in range(1, m + 1):
            cur = dp[j]
            dp[j] = min(dp[j] + 1, dp[j - 1] + 1, prev + (a[i - 1] != b[j - 1]))
            prev = cur
    return dp[m]

def lcs_len(a, b):
    n, m = len(a), len(b)
    dp = [0] * (m + 1)
    for i in range(1, n + 1):
        prev = 0
        for j in range(1, m + 1):
            cur = dp[j]
            dp[j] = prev + 1 if a[i - 1] == b[j - 1] else max(dp[j], dp[j - 1])
            prev = cur
    return dp[m]

def is_subsequence(required, actual):
    """必经动作断言：required 必须作为子序列出现在 actual 中。"""
    it = iter(actual)
    return all(any(x == y for y in it) for x in required)

REF = ['get_policy', 'search_orders', 'update_order', 'send_email']
A   = ['get_policy', 'search_orders', 'update_order', 'send_email']              # 完全一致
B   = ['get_policy', 'list_flights', 'search_orders', 'search_orders',
       'update_order', 'send_email']                                            # 多绕了两步
C   = ['search_orders', 'update_order', 'send_email']                            # 漏了必经动作

for name, seq in [('A 完全一致', A), ('B 多绕两步', B), ('C 漏了 get_policy', C)]:
    print(f'{name:<18} 编辑距离 {edit_distance(REF, seq)} | LCS {lcs_len(REF, seq)} | '
          f'必经动作断言 {is_subsequence(["get_policy", "update_order"], seq)}')

assert edit_distance(REF, A) == 0
assert edit_distance(REF, B) > 0 and is_subsequence(['get_policy', 'update_order'], B)
assert not is_subsequence(['get_policy', 'update_order'], C)
print('\n✅ 关键对比在 B 这一行：编辑距离说它「偏离了」，必经动作断言说它「合规」。')
print('   B 只是多做了两步探索——把编辑距离当主指标，就是在惩罚探索。断言才是对的工具。')

## 5 · 失败模式自动分类器（互斥、有优先级）

In [ ]:
def classify_failure(traj, median_steps, redundancy_thresh=0.35):
    """自上而下第一个命中即分类——必须互斥，否则比例加起来会超过 100%。"""
    if traj['outcome']['score'] == 1:
        return 'success'
    steps = traj['steps']
    n = len(steps)
    err_codes = [s['error_code'] for s in steps if s['status'] != 'ok']
    if detect_exact_loop(traj)[0] or redundancy_rate(traj) > redundancy_thresh:
        return 'loop'
    if len(err_codes) >= 3 and len(set(err_codes)) == 1:
        return 'tool_misuse'
    if n >= traj['max_steps']:
        return 'budget_exhausted'
    if n < 0.5 * median_steps and not err_codes:
        return 'early_quit'
    if not err_codes:
        return 'hallucinated_completion'
    return 'genuine_incapability'

misusers = [simulate_trajectory(rng, 'tool_misuser') for _ in range(80)]
ALL = healthy + loopers + quitters + misusers
med = np.median([len(t['steps']) for t in ALL])

cnt = Counter(classify_failure(t, med) for t in ALL)
total_fail = sum(v for k, v in cnt.items() if k != 'success')
print(f'共 {len(ALL)} 条轨迹，失败 {total_fail} 条。失败原因分解:')
for k, v in cnt.most_common():
    if k == 'success':
        continue
    print(f'  {k:<26} {v:>4} ({v/total_fail:5.1%})')

assert sum(cnt.values()) == len(ALL), '七分类必须互斥且穷尽'
assert cnt['loop'] > 0 and cnt['early_quit'] > 0
non_capability = 1 - cnt['genuine_incapability'] / total_fail
print(f'\n「不是模型不够强」的失败占比: {non_capability:.0%}')
print('✅ 这个数字是本模块最有行动价值的产出——')
print('   它告诉你：在换更强的模型之前，还有这么大一块是 harness / 提示 / 工具描述的问题。')

## 6 · 过程标注一致性：Cohen's kappa 与「换个问法」

In [ ]:
def cohen_kappa(a, b):
    a, b = np.asarray(a), np.asarray(b)
    cats = sorted(set(a.tolist()) | set(b.tolist()))
    po = float((a == b).mean())
    pe = sum((a == c).mean() * (b == c).mean() for c in cats)
    return (po - pe) / (1 - pe) if abs(1 - pe) > 1e-12 else float('nan')

rng = np.random.default_rng(19)
N = 400
# 问法一「这一步好不好」：主观，两个标注员各有偏好
truth = rng.integers(0, 3, size=N)
ann1_vague = np.where(rng.random(N) < 0.55, truth, rng.integers(0, 3, size=N))
ann2_vague = np.where(rng.random(N) < 0.55, truth, rng.integers(0, 3, size=N))
# 问法二「这一步是否让任务离目标更近」：二元、可判定，一致性显著提高
truth_b = (truth > 0).astype(int)
ann1_sharp = np.where(rng.random(N) < 0.90, truth_b, 1 - truth_b)
ann2_sharp = np.where(rng.random(N) < 0.90, truth_b, 1 - truth_b)

k_vague = cohen_kappa(ann1_vague, ann2_vague)
k_sharp = cohen_kappa(ann1_sharp, ann2_sharp)
print(f'问法一「这一步好不好」（三档）  kappa = {k_vague:.3f}')
print(f'问法二「是否离目标更近」（二元） kappa = {k_sharp:.3f}')
assert k_sharp > k_vague + 0.2
print('\n判读标准（沿用 C10-02）：<0.4 差 | 0.4-0.6 中等 | 0.6-0.8 良好 | >0.8 优秀')
print(f'✅ 只是换了个问法，kappa 从 {k_vague:.2f} 提到 {k_sharp:.2f}——')
print('   kappa 低时，第一反应应该是「标注问题问得不好」，而不是「再培训标注员」。')

## 7 · Goodhart 演示：优化平均步数，把成功率优化没了

In [ ]:
def agent_with_patience(patience, n_tasks=800, seed=0):
    """patience 越大越不容易放弃：每一步继续尝试的概率与 patience 相关。
    返回 (成功率, 全部轨迹平均步数, 成功轨迹中位步数, 提前放弃率)。"""
    rng = np.random.default_rng(seed)
    steps_all, succ, quits = [], [], 0
    for _ in range(n_tasks):
        need = int(rng.integers(4, 20))            # 这个任务真正需要的步数
        k = 0
        while k < need and rng.random() < patience:
            k += 1
        ok = (k >= need)
        if not ok:
            quits += 1
        steps_all.append(k + 1)
        succ.append(float(ok))
    steps_all = np.array(steps_all)
    succ = np.array(succ)
    med_succ = float(np.median(steps_all[succ == 1])) if succ.sum() else float('nan')
    return float(succ.mean()), float(steps_all.mean()), med_succ, quits / n_tasks

print(f"{'patience':>9}{'成功率':>10}{'全体平均步数':>14}{'成功轨迹中位步数':>18}{'提前放弃率':>12}")
rows = []
for pat in [0.99, 0.95, 0.90, 0.80, 0.70]:
    r = agent_with_patience(pat, seed=3)
    rows.append((pat,) + r)
    print(f'{pat:>9.2f}{r[0]:>10.1%}{r[1]:>14.2f}{r[2]:>18.1f}{r[3]:>12.1%}')

# 「优化平均步数」会选中最差的那一行
best_by_mean_steps = min(rows, key=lambda r: r[2])
best_by_success = max(rows, key=lambda r: r[1])
print(f'\n按「全体平均步数最小」选出的配置: patience={best_by_mean_steps[0]:.2f} '
      f'(成功率仅 {best_by_mean_steps[1]:.1%})')
print(f'按「成功率最大」选出的配置:       patience={best_by_success[0]:.2f} '
      f'(成功率 {best_by_success[1]:.1%})')
assert best_by_mean_steps[0] != best_by_success[0]
assert best_by_mean_steps[1] < best_by_success[1]
print('\n✅ Goodhart 现场：优化「平均步数」这个指标，选出的是成功率最低的配置。')
print('   修正：效率只在成功轨迹上算 + 成本必须与成功率成对报告 + 单独监控提前放弃率。')

## ✏️ 练习 1：加权无效动作率（区分首次探索与重复犯错）

实现 `weighted_invalid_rate(traj, repeat_penalty=3.0)`：
无效动作里，**第一次**出现某个 `(tool, error_code)` 组合权重记 1.0，
**之后每次重复**记 `repeat_penalty`。返回 `加权无效动作数 / 总步数`。

In [ ]:
def weighted_invalid_rate(traj, repeat_penalty=3.0):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
def mk(tool, status, code):
    return {'tool': tool, 'status': status, 'error_code': code}

t1 = {'steps': [mk('a', 'ok', None), mk('a', 'tool_error', 'E1'), mk('a', 'tool_error', 'E1')]}
# 一次首犯(1.0) + 一次重复(3.0) = 4.0 / 3 步
assert abs(weighted_invalid_rate(t1) - 4.0 / 3) < 1e-12
t2 = {'steps': [mk('a', 'tool_error', 'E1'), mk('b', 'tool_error', 'E2')]}
assert abs(weighted_invalid_rate(t2) - 1.0) < 1e-12       # 两次都是首犯
t3 = {'steps': [mk('a', 'ok', None), mk('b', 'ok', None)]}
assert weighted_invalid_rate(t3) == 0.0
print(f'两次同样的错: {weighted_invalid_rate(t1):.3f} | 两个不同的错: {weighted_invalid_rate(t2):.3f}')
print('✅ 练习 1 通过：探索性试错和「反复撞同一堵墙」不该同权——')
print('   前者是 agent 在获取信息，后者是它没有从反馈里学到东西。')

## ✏️ 练习 2：恢复事件级的恢复率

实现 `event_recovery_rate(traj)`：以**事件**而非轨迹为单位。
对轨迹中每一次 `status != 'ok'` 的步骤，看它的<strong>下一步</strong>：
若下一步 `status == 'ok'` 且 `tool` 或 `args_hash` 与出错那步不同，记为一次成功恢复。
返回 `(成功恢复数, 错误事件总数, 恢复率)`；无错误时恢复率返回 `float('nan')`。

In [ ]:
def event_recovery_rate(traj):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
def mk2(tool, ah, status):
    return {'tool': tool, 'args_hash': ah, 'status': status}

good = {'steps': [mk2('a', 'h1', 'tool_error'), mk2('b', 'h2', 'ok')]}
assert event_recovery_rate(good) == (1, 1, 1.0)
stuck = {'steps': [mk2('a', 'h1', 'tool_error'), mk2('a', 'h1', 'ok')]}   # 换都没换，不算恢复
assert event_recovery_rate(stuck) == (0, 1, 0.0)
clean = {'steps': [mk2('a', 'h1', 'ok')]}
n_rec, n_err, rate = event_recovery_rate(clean)
assert (n_rec, n_err) == (0, 0) and math.isnan(rate)
last_step_err = {'steps': [mk2('a', 'h1', 'ok'), mk2('b', 'h2', 'tool_error')]}
assert event_recovery_rate(last_step_err) == (0, 1, 0.0)   # 出错就没有下一步了
print('全部四种情形通过：正常恢复 / 原地重试 / 无错误 / 末步出错')
print('✅ 练习 2 通过：事件级恢复率比轨迹级更灵敏——')
print('   一条轨迹出了五次错恢复了四次，轨迹级只能记「成功」或「失败」一个 bit。')

## ✏️ 练习 3：成本-成功率的分层报告

实现 `stratified_efficiency(trajs)`：返回一个字典，包含
`success_rate`、`median_steps_success`（成功轨迹的中位步数）、
`p90_steps_success`、`early_quit_rate`（未成功且步数 < 全体中位数一半的比例）。
这四个数字就是第 7 节三条修正规则的可执行版本。

In [ ]:
def stratified_efficiency(trajs):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
rep = stratified_efficiency(ALL)
assert set(rep) == {'success_rate', 'median_steps_success', 'p90_steps_success', 'early_quit_rate'}
assert 0 <= rep['success_rate'] <= 1 and 0 <= rep['early_quit_rate'] <= 1
assert rep['p90_steps_success'] >= rep['median_steps_success']
for k, v in rep.items():
    print(f'  {k:<24} {v:.3f}')
print('✅ 练习 3 通过：这四行就是可以直接贴进报告的效率部分——')
print('   注意里面没有「平均步数」，这是刻意的。')

## ✏️ 练习 4：过程分与结果分的相关性

实现 `process_outcome_corr(process_scores, outcomes)`：返回皮尔逊相关系数
（纯 numpy，不用 scipy）。用它验证「过程分高但结果失败」的样本确实存在，
即相关系数显著大于 0 但远小于 1。

In [ ]:
def process_outcome_corr(process_scores, outcomes):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
x = np.array([1.0, 2.0, 3.0, 4.0])
assert abs(process_outcome_corr(x, x) - 1.0) < 1e-9
assert abs(process_outcome_corr(x, -x) + 1.0) < 1e-9

rng = np.random.default_rng(31)
n = 500
proc = rng.uniform(0, 1, n)
out = (rng.random(n) < np.clip(proc * 0.8 + 0.05, 0, 1)).astype(float)
r = process_outcome_corr(proc, out)
print(f'过程分与结果分的相关系数 r = {r:.3f}')
assert 0.2 < r < 0.9
good_proc_fail = ((proc > 0.8) & (out == 0)).sum()
print(f'过程分 >0.8 但最终失败的样本: {good_proc_fail} 条')
assert good_proc_fail > 0
print('✅ 练习 4 通过：相关但远非等同——这就是「过程分不能替代结果分作主指标」的实证形式。')
print('   过程好但没做完，产品价值是 0；而只看结果分，你不知道它差在哪一步。')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def weighted_invalid_rate(traj, repeat_penalty=3.0):
    steps = traj['steps']
    if not steps:
        return 0.0
    seen, total = set(), 0.0
    for s in steps:
        if s['status'] != 'ok':
            key = (s['tool'], s['error_code'])
            total += repeat_penalty if key in seen else 1.0
            seen.add(key)
    return total / len(steps)

In [ ]:
# 练习 2 参考答案
def event_recovery_rate(traj):
    steps = traj['steps']
    n_err = n_rec = 0
    for i, s in enumerate(steps):
        if s['status'] == 'ok':
            continue
        n_err += 1
        if i + 1 < len(steps):
            nxt = steps[i + 1]
            changed = (nxt['tool'] != s['tool']) or (nxt['args_hash'] != s['args_hash'])
            if nxt['status'] == 'ok' and changed:
                n_rec += 1
    rate = n_rec / n_err if n_err else float('nan')
    return (n_rec, n_err, rate)

In [ ]:
# 练习 3 参考答案
def stratified_efficiency(trajs):
    lens = np.array([len(t['steps']) for t in trajs], dtype=float)
    succ = np.array([t['outcome']['score'] for t in trajs], dtype=float)
    med_all = float(np.median(lens))
    s_lens = lens[succ == 1]
    return {
        'success_rate': float(succ.mean()),
        'median_steps_success': float(np.median(s_lens)) if s_lens.size else float('nan'),
        'p90_steps_success': float(np.percentile(s_lens, 90)) if s_lens.size else float('nan'),
        'early_quit_rate': float(((succ == 0) & (lens < 0.5 * med_all)).mean()),
    }

In [ ]:
# 练习 4 参考答案
def process_outcome_corr(process_scores, outcomes):
    x = np.asarray(process_scores, dtype=float)
    y = np.asarray(outcomes, dtype=float)
    xc, yc = x - x.mean(), y - y.mean()
    denom = math.sqrt(float((xc ** 2).sum()) * float((yc ** 2).sum()))
    return float((xc * yc).sum() / denom) if denom else float('nan')

---
## 🧪 真实工程胶囊：把轨迹分析接到真实 agent 上

In [ ]:
RECIPE = r'''
# ══════════════════════════════════════════════════════════════════
# A. 用 OpenTelemetry GenAI 语义约定记轨迹（别自造字段名）
# ══════════════════════════════════════════════════════════════════
from opentelemetry import trace
tracer = trace.get_tracer("agent.eval")

with tracer.start_as_current_span("agent.run") as run_span:
    run_span.set_attribute("gen_ai.system", "anthropic")
    run_span.set_attribute("gen_ai.request.model", "claude-sonnet-5")
    run_span.set_attribute("eval.task_id", task_id)
    run_span.set_attribute("eval.attempt", attempt)          # ← pass^k 需要
    run_span.set_attribute("eval.scaffold_version", SCAFFOLD_VERSION)
    for i, step in enumerate(agent_loop()):
        with tracer.start_as_current_span("gen_ai.tool.execute") as sp:
            sp.set_attribute("gen_ai.tool.name", step.tool)
            sp.set_attribute("eval.args_hash", step.args_hash)   # ← 冗余率需要
            sp.set_attribute("eval.status", step.status)         # ← 恢复率需要
            sp.set_attribute("gen_ai.usage.input_tokens", step.tokens_in)
            sp.set_attribute("gen_ai.usage.output_tokens", step.tokens_out)
# 用现成的 trace 查看器（Jaeger / Grafana Tempo / Phoenix / LangSmith）直接看，
# 不需要自己写 UI。这是沿用标准字段名的唯一理由，也是足够的理由。

# ══════════════════════════════════════════════════════════════════
# B. 主动注入错误，测恢复力（离线评测最容易漏的一步）
# ══════════════════════════════════════════════════════════════════
class FlakyTool:
    # 按概率注入真实感的失败：超时、限流、参数错误、空结果
    def __init__(self, inner, p=0.15, seed=0):
        self.inner, self.p, self.rng = inner, p, random.Random(seed)
    def __call__(self, **kw):
        if self.rng.random() < self.p:
            raise random.choice([TimeoutError("upstream timeout"),
                                 RuntimeError("429 rate limited"),
                                 ValueError("invalid argument: order_id")])
        return self.inner(**kw)
# 报告规范：注入率 0% / 10% / 25% 三档各跑一遍，报告三条成功率曲线。
# 曲线的斜率就是「恢复力」，比任何单点数字都有信息量。

# ══════════════════════════════════════════════════════════════════
# C. 失败分类的落地：从 trace 直接产出周报
# ══════════════════════════════════════════════════════════════════
# 每周跑一次，输出：
#   1) 失败分解饼图（七类，互斥）
#   2) 「非能力问题」占比的时间序列   ← 这条曲线在下降，说明 harness 在变好
#   3) 提前放弃率的时间序列          ← 这条在上升，说明有人在优化步数，要拦
#   4) 出错轨迹 vs 无错轨迹的条件成功率落差 ← 这条预测线上表现
'''
print(RECIPE)

### 小结

| 你学到的 | 一句话 | 用在哪 |
|---|---|---|
| 三件结果层看不见的事 | 钱花在哪、卡在哪、能不能自救 | 归因与改进 |
| schema 是事前决定 | 没记 `args_hash` / `status` / `attempt`，指标永远算不出来 | 写第一个 agent 之前 |
| 六个指标 | 步数分位数、工具精确/召回、无效动作、冗余、恢复 | 周报固定项 |
| 轨迹比较做断言不做打分 | 必经动作用 LCS 子序列；编辑距离只做人工复核分诊 | 任务设计 |
| 失败分类学 | 归因为「模型不够强」之前，先排除前六类 | 决定下一步修什么 |
| 过程标注 kappa | kappa 低先怀疑问法，不是标注员 | 过程级评测 |
| Goodhart | 「平均步数」不该出现在任何报告里 | 指标引入的第一天 |

下一模块：**04 · 可靠性与统计**——pass@k 与 pass^k、方差从哪来、
两个 agent 差 3 个点到底算不算差、以及要跑几个 seed 才够。